<a href="https://colab.research.google.com/github/boradenihal15-beep/FlyRank_Ai_ML/blob/main/Works/Notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:
%pip -q install duckdb huggingface_hub


In [13]:
import os, getpass

# CI and power users set HF_TOKEN in the environment; everyone else gets the safe prompt.
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


Paste your Hugging Face READ token (hf_...): ··········


In [14]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [15]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features):,} content items with enough history')
features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

111,247 content items with enough history


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30
0,client_62f4a7e64f5e0096,content_0dac238195631de4,219.0,251.0,0.0,3.468159
1,client_62f4a7e64f5e0096,content_f6116743b00afc2d,995.0,4786.0,2.0,25.024091
2,client_62f4a7e64f5e0096,content_332f337f995e9781,103.0,177.0,0.0,18.600206
3,client_62f4a7e64f5e0096,content_82053c8b9d8a4811,736.0,1518.0,2.0,9.526655
4,client_62f4a7e64f5e0096,content_742939be32c206d2,269.0,335.0,3.0,7.904483


In [16]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']
data = features.merge(qsignals, on='content_hash_id', how='left')
print(f'joined: {len(data):,} rows')
data.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

joined: 111,247 rows


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share
0,client_62f4a7e64f5e0096,content_0dac238195631de4,219.0,251.0,0.0,3.468159,15.0,0.144623,0.665019,79.0,308.0,0.256494
1,client_62f4a7e64f5e0096,content_f6116743b00afc2d,995.0,4786.0,2.0,25.024091,101.0,0.037423,0.178737,15557.0,18432.0,0.844021
2,client_62f4a7e64f5e0096,content_332f337f995e9781,103.0,177.0,0.0,18.600206,3.0,0.215054,0.623656,25.0,60.0,0.416667
3,client_62f4a7e64f5e0096,content_82053c8b9d8a4811,736.0,1518.0,2.0,9.526655,16.0,0.032740,0.717915,473.0,952.0,0.496849
4,client_62f4a7e64f5e0096,content_742939be32c206d2,269.0,335.0,3.0,7.904483,8.0,0.224066,0.630705,30.0,140.0,0.214286


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

## Build the Feature Vector

The feature vector contains historical search performance metrics that are available before the prediction period. These features are engineered from the warehouse data and are intended to help predict whether a content page's search impressions will decline.

The selected features are numeric, require no categorical encoding, and missing values are filled with zero where necessary.

In [17]:
feature_cols = [
    "imp_prev30",
    "clk_last30",
    "pos_last30",
    "visible_queries",
    "rare_share",
    "anon_share",
    "top_query_share"
]

X = data[feature_cols].copy()

# Fill missing values
X = X.fillna(0)

print("Feature Vector Shape:", X.shape)

X.head()

Feature Vector Shape: (111247, 7)


,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_share
0,251.0,0.0,3.468159,15.0,0.144623,0.665019,0.256494
1,4786.0,2.0,25.024091,101.0,0.037423,0.178737,0.844021
2,177.0,0.0,18.600206,3.0,0.215054,0.623656,0.416667
3,1518.0,2.0,9.526655,16.0,0.032740,0.717915,0.496849
4,335.0,3.0,7.904483,8.0,0.224066,0.630705,0.214286


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

## Feature Notes

| Feature | Meaning | Missing Values | Categorical | Available Before Prediction |
|---------|---------|---------------|-------------|-----------------------------|
| imp_prev30 | Impressions during previous 30 days | Filled with 0 | No | Yes |
| clk_last30 | Clicks during previous period | Filled with 0 | No | Yes |
| pos_last30 | Average search position | Filled with 0 | No | Yes |
| visible_queries | Number of visible queries | Filled with 0 | No | Yes |
| rare_share | Share of rare queries | Filled with 0 | No | Yes |
| anon_share | Share of anonymized queries | Filled with 0 | No | Yes |
| top_query_share | Share contributed by top query | Filled with 0 | No | Yes |

All selected features are numerical. No categorical encoding was required. Missing values were handled using zero filling where applicable.

In [18]:
X.describe()

,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_share
count,111247.000000,111247.000000,111247.000000,111247.000000,111247.000000,111247.000000,111247.000000
mean,2290.660665,6.972350,20.304596,20.904806,0.114378,0.608655,0.360609
std,6942.053859,32.517226,17.920511,51.022378,0.118800,0.274745,0.269234
min,100.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,231.000000,0.000000,7.504702,3.000000,0.030920,0.477889,0.169231
50%,565.000000,1.000000,13.396795,8.000000,0.076923,0.691169,0.294533
75%,1801.000000,5.000000,27.650000,21.000000,0.158367,0.816300,0.500000
max,585502.000000,3817.000000,579.000000,7889.000000,0.890710,0.996458,1.000000


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

## Leakage Hunt

Every feature was reviewed to ensure it would be available before prediction time.

The following types of leakage were checked:

- Future-window leakage
- Label leakage
- Product-generated features
- Private identifiers
- URLs and raw queries

In [19]:
safe_features = [
    "imp_prev30",
    "clk_last30",
    "pos_last30",
    "visible_queries",
    "rare_share",
    "anon_share",
    "top_query_share"
]

unsafe_features = [
    "imp_last30",
    "client_hash_id",
    "content_hash_id",
    "health_score",
    "raw_query",
    "url"
]

print("Safe Features:")
for f in safe_features:
    print("✓", f)

print("\nUnsafe Features:")
for f in unsafe_features:
    print("✗", f)

Safe Features:
✓ imp_prev30
✓ clk_last30
✓ pos_last30
✓ visible_queries
✓ rare_share
✓ anon_share
✓ top_query_share

Unsafe Features:
✗ imp_last30
✗ client_hash_id
✗ content_hash_id
✗ health_score
✗ raw_query
✗ url


### Leakage Assessment

Unsafe fields were excluded because:

- imp_last30 overlaps the prediction window.
- client_hash_id is a private identifier.
- content_hash_id is only an identifier.
- raw_query contains confidential search terms.
- URL contains private client information.
- health_score is a product-generated feature that may leak internal knowledge.

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

## Excluded Fields

| Field | Reason for Exclusion |
|-------|----------------------|
| client_hash_id | Private identifier |
| content_hash_id | Identifier only |
| imp_last30 | Contains future information relative to prediction |
| raw_query | Private search data |
| URL | Client confidential information |
| health_score | Product-generated feature |

These fields were intentionally excluded to avoid privacy issues and information leakage.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.